<a href="https://colab.research.google.com/github/Louaighoul/AI-Camp-Sentence-Similarity-Challenge/blob/main/Copy_of_first_competition.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# Fix for Weights & Biases API key error
# This command will prompt you to enter your Weights & Biases API key interactively.
# Please ensure the API key is at least 40 characters long.
!wandb login

wandb: Logging into wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: You can find your API key in your browser here: https://wandb.ai/authorize?ref=models
wandb: Paste an API key from your profile and hit enter, or press ctrl+c to quit: 
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: ghoulilouai (ghoulilouai-ecole-nationale-sup-rieure-d-informatique) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


In [ ]:
import pandas as pd


df_train=pd.read_csv("/content/train.csv")
df_test=pd.read_csv("/content/test.csv")

In [ ]:
# Install the core library
!pip install sentence-transformers pandas numpy

import pandas as pd
import numpy as np
from sentence_transformers import SentenceTransformer, InputExample, losses
from torch.utils.data import DataLoader
import os

# --- Configuration Constants ---
# MODEL_NAME is a popular, fast model optimized for semantic similarity
MODEL_NAME = 'all-MiniLM-L6-v2'
TRAIN_BATCH_SIZE = 16
NUM_EPOCHS = 3

In [ ]:
# --- Define File Paths (Assuming files are in the current working directory) ---


try:
    # 1. Load data


    # 2. Normalize the target score from [0, 5] to the required [0, 1]
    df_train['normalized_score'] = df_train['similarity_score'] / 5.0

    # 3. Create a list of InputExample objects for the training data
    train_examples = []
    for index, row in df_train.iterrows():
        # InputExample format: (Sentence 1, Sentence 2, Label [normalized])
        train_examples.append(InputExample(
            texts=[row['sentence1'], row['sentence2']],
            label=row['normalized_score']
        ))

    # 4. Define the DataLoader
    train_dataloader = DataLoader(
        train_examples,
        shuffle=True,
        batch_size=TRAIN_BATCH_SIZE
    )

    print(f"✅ Data loaded and {len(train_examples)} training examples prepared.")

except FileNotFoundError:
    print(f"❌ ERROR: train.csv or test.csv not found. Please ensure the files are in the current directory.")
    # Stop execution if files aren't found
    raise

✅ Data loaded and 5749 training examples prepared.


In [ ]:
# 1. Load the Pre-trained Sentence Transformer Model
# This model acts as the shared-weight Siamese base
try:
    model = SentenceTransformer(MODEL_NAME)
    print(f"✅ Sentence Transformer model '{MODEL_NAME}' loaded.")
except Exception as e:
    print(f"❌ ERROR loading model. Network issue likely persists. Fix the network and retry: {e}")
    # Stop execution if model download fails
    raise

# 2. Define the Loss Function (Standard for STS fine-tuning)
train_loss = losses.CosineSimilarityLoss(model)

# 3. Fine-Tune the Model
WARMUP_STEPS = int(len(train_dataloader) * NUM_EPOCHS * 0.1) # 10% of total steps for learning rate warm-up

print("\nStarting S-BERT Fine-Tuning...")

model.fit(
    train_objectives=[(train_dataloader, train_loss)],
    epochs=NUM_EPOCHS,
    warmup_steps=WARMUP_STEPS,
    output_path='sbert_sts_model', # Directory to save the final model
    show_progress_bar=True,
    optimizer_params={'lr': 2e-5} # Optimal learning rate for S-BERT
)

print("\n✅ S-BERT Model Fine-Tuning Complete.")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

✅ Sentence Transformer model 'all-MiniLM-L6-v2' loaded.

Starting S-BERT Fine-Tuning...


Computing widget examples:   0%|          | 0/1 [00:00<?, ?example/s]

/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: Currently logged in as: ghoulilouai (ghoulilouai-ecole-nationale-sup-rieure-d-informatique) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


Step,Training Loss
500,0.021800
1000,0.014400



✅ S-BERT Model Fine-Tuning Complete.


In [ ]:
from sentence_transformers.util import cos_sim

# 1. Get the two sentence lists from the test set
sentences1 = df_test['sentence1'].tolist()
sentences2 = df_test['sentence2'].tolist()

print("\nGenerating Embeddings and Predictions for Test Sentences...")

# 2. Generate embeddings for both sets of sentences
embeddings1 = model.encode(sentences1, convert_to_tensor=True, show_progress_bar=True)
embeddings2 = model.encode(sentences2, convert_to_tensor=True, show_progress_bar=True)

# 3. Calculate Cosine Similarity
# The result is a matrix; we take the diagonal since we want similarity between pair i, i
similarity_scores_tensor = cos_sim(embeddings1, embeddings2)
predicted_scores = np.diag(similarity_scores_tensor.cpu().numpy())

# 4. Create Submission File
submission_df = pd.DataFrame({
    'id': df_test['id'],
    'similarity_score': predicted_scores
})

# Ensure the score is clipped to the required [0, 1] range
submission_df['similarity_score'] = submission_df['similarity_score'].clip(lower=0.0, upper=1.0)

# Save the submission file
submission_df.to_csv('submission.csv', index=False)

print("\n✅ Submission file 'submission.csv' created.")
print("Submission Head:")
print(submission_df.head())


Generating Embeddings and Predictions for Test Sentences...


Batches:   0%|          | 0/44 [00:00<?, ?it/s]

Batches:   0%|          | 0/44 [00:00<?, ?it/s]


✅ Submission file 'submission.csv' created.
Submission Head:
   id  similarity_score
0   0          0.754266
1   1          0.813897
2   2          0.946101
3   3          0.921262
4   4          0.232742
